In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw/primary")
PROCESSED_DIR = Path("../data/processed/primary")

files = {
    2008: RAW_DIR / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv",
    2009: RAW_DIR / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv",
    2010: RAW_DIR / "DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv"
}

In [2]:
dfs = {}

for year, file in files.items():
    df = pd.read_csv(file)
    dfs[year] = df

    print(f"{year}: {df.shape}")

2008: (116352, 32)
2009: (114538, 32)
2010: (112754, 32)


In [3]:
columns_2008 = set(dfs[2008].columns)
columns_2009 = set(dfs[2009].columns)
columns_2010 = set(dfs[2010].columns)

print("2008 == 2009:", columns_2008 == columns_2009)
print("2009 == 2010:", columns_2009 == columns_2010)

2008 == 2009: True
2009 == 2010: True


In [4]:
print(
    list(dfs[2008].columns)
    == list(dfs[2009].columns)
)

print(
    list(dfs[2009].columns)
    == list(dfs[2010].columns)
)

True
True


In [5]:
for year, df in dfs.items():
    print(
        year,
        "missing IDs:",
        df["DESYNPUF_ID"].isna().sum()
    )

2008 missing IDs: 0
2009 missing IDs: 0
2010 missing IDs: 0


In [6]:
for year, df in dfs.items():
    print(
        year,
        "duplicate IDs:",
        df["DESYNPUF_ID"].duplicated().sum()
    )

2008 duplicate IDs: 0
2009 duplicate IDs: 0
2010 duplicate IDs: 0


In [7]:
for year, df in dfs.items():
    print("\n", year)
    print(df.dtypes)


 2008
DESYNPUF_ID                  object
BENE_BIRTH_DT                 int64
BENE_DEATH_DT               float64
BENE_SEX_IDENT_CD             int64
BENE_RACE_CD                  int64
BENE_ESRD_IND                object
SP_STATE_CODE                 int64
BENE_COUNTY_CD                int64
BENE_HI_CVRAGE_TOT_MONS       int64
BENE_SMI_CVRAGE_TOT_MONS      int64
BENE_HMO_CVRAGE_TOT_MONS      int64
PLAN_CVRG_MOS_NUM             int64
SP_ALZHDMTA                   int64
SP_CHF                        int64
SP_CHRNKIDN                   int64
SP_CNCR                       int64
SP_COPD                       int64
SP_DEPRESSN                   int64
SP_DIABETES                   int64
SP_ISCHMCHT                   int64
SP_OSTEOPRS                   int64
SP_RA_OA                      int64
SP_STRKETIA                   int64
MEDREIMB_IP                 float64
BENRES_IP                   float64
PPPYMT_IP                   float64
MEDREIMB_OP                 float64
BENRES_OP            

In [8]:
for year, df in dfs.items():
    df["YEAR"] = year

In [9]:
beneficiary = pd.concat(
    dfs.values(),
    ignore_index=True
)

In [10]:
print("Shape:", beneficiary.shape)

Shape: (343644, 33)


In [11]:
print(
    beneficiary["YEAR"]
    .value_counts()
    .sort_index()
)

YEAR
2008    116352
2009    114538
2010    112754
Name: count, dtype: int64


In [12]:
print(
    "Total unique beneficiaries:",
    beneficiary["DESYNPUF_ID"].nunique()
)

Total unique beneficiaries: 116352


In [13]:
years_per_beneficiary = (
    beneficiary
    .groupby("DESYNPUF_ID")["YEAR"]
    .nunique()
)

print(
    years_per_beneficiary.value_counts()
    .sort_index()
)

YEAR
1      1814
2      1784
3    112754
Name: count, dtype: int64


In [14]:
missing = (
    beneficiary
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

BENE_DEATH_DT               338183
DESYNPUF_ID                      0
PPPYMT_IP                        0
SP_ISCHMCHT                      0
SP_OSTEOPRS                      0
SP_RA_OA                         0
SP_STRKETIA                      0
MEDREIMB_IP                      0
BENRES_IP                        0
MEDREIMB_OP                      0
SP_DEPRESSN                      0
BENRES_OP                        0
PPPYMT_OP                        0
MEDREIMB_CAR                     0
BENRES_CAR                       0
PPPYMT_CAR                       0
SP_DIABETES                      0
SP_COPD                          0
BENE_BIRTH_DT                    0
SP_CNCR                          0
SP_CHRNKIDN                      0
SP_CHF                           0
SP_ALZHDMTA                      0
PLAN_CVRG_MOS_NUM                0
BENE_HMO_CVRAGE_TOT_MONS         0
BENE_SMI_CVRAGE_TOT_MONS         0
BENE_HI_CVRAGE_TOT_MONS          0
BENE_COUNTY_CD                   0
SP_STATE_CODE       

In [21]:
beneficiary["BENE_ESRD_IND"] = (
    beneficiary["BENE_ESRD_IND"]
    .astype(str)
    .str.strip()
    .map({"0": 0, "Y": 1})
)

In [22]:
print(
    beneficiary["BENE_SEX_IDENT_CD"]
    .value_counts(dropna=False)
)

print(
    beneficiary["BENE_RACE_CD"]
    .value_counts(dropna=False)
)

print(
    beneficiary["BENE_ESRD_IND"]
    .value_counts(dropna=False)
)

BENE_SEX_IDENT_CD
2    190059
1    153585
Name: count, dtype: int64
BENE_RACE_CD
1    284514
2     36459
3     14591
5      8080
Name: count, dtype: int64
BENE_ESRD_IND
0    316510
1     27134
Name: count, dtype: int64


In [16]:
beneficiary["BENE_BIRTH_DT"] = pd.to_datetime(
    beneficiary["BENE_BIRTH_DT"],
    format="%Y%m%d",
    errors="coerce"
)

beneficiary["BENE_DEATH_DT"] = pd.to_datetime(
    beneficiary["BENE_DEATH_DT"],
    format="%Y%m%d",
    errors="coerce"
)

In [17]:
print(beneficiary[
    ["BENE_BIRTH_DT", "BENE_DEATH_DT"]
].dtypes)

BENE_BIRTH_DT    datetime64[ns]
BENE_DEATH_DT    datetime64[ns]
dtype: object


In [24]:
print(
    "Missing birth dates:",
    beneficiary["BENE_BIRTH_DT"].isna().sum()
)

print(
    "Missing death dates:",
    beneficiary["BENE_DEATH_DT"].isna().sum()
)

Missing birth dates: 0
Missing death dates: 338183


In [25]:
print(
    "Birth dates after death dates:",
    (
        beneficiary["BENE_DEATH_DT"].notna()
        & (
            beneficiary["BENE_DEATH_DT"]
            < beneficiary["BENE_BIRTH_DT"]
        )
    ).sum()
)

Birth dates after death dates: 0


In [26]:
print("Birth date range:")
print(beneficiary["BENE_BIRTH_DT"].min())
print(beneficiary["BENE_BIRTH_DT"].max())

print("\nDeath date range:")
print(beneficiary["BENE_DEATH_DT"].min())
print(beneficiary["BENE_DEATH_DT"].max())

Birth date range:
1909-01-01 00:00:00
1983-12-01 00:00:00

Death date range:
2008-01-01 00:00:00
2010-12-01 00:00:00


In [27]:
missing_report = pd.DataFrame({
    "missing_count": beneficiary.isna().sum(),
    "missing_percent": (
        beneficiary.isna().mean() * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    "missing_percent",
    ascending=False
)

print(missing_report)

                          missing_count  missing_percent
BENE_DEATH_DT                    338183            98.41
DESYNPUF_ID                           0             0.00
PPPYMT_IP                             0             0.00
SP_ISCHMCHT                           0             0.00
SP_OSTEOPRS                           0             0.00
SP_RA_OA                              0             0.00
SP_STRKETIA                           0             0.00
MEDREIMB_IP                           0             0.00
BENRES_IP                             0             0.00
MEDREIMB_OP                           0             0.00
SP_DEPRESSN                           0             0.00
BENRES_OP                             0             0.00
PPPYMT_OP                             0             0.00
MEDREIMB_CAR                          0             0.00
BENRES_CAR                            0             0.00
PPPYMT_CAR                            0             0.00
SP_DIABETES                    

In [28]:
numeric_cols = beneficiary.select_dtypes(
    include="number"
).columns

print(
    beneficiary[numeric_cols].describe().T
)

                             count         mean          std     min     25%  \
BENE_SEX_IDENT_CD         343644.0     1.553069     0.497176     1.0     1.0   
BENE_RACE_CD              343644.0     1.285065     0.755564     1.0     1.0   
BENE_ESRD_IND             343644.0     0.078960     0.269676     0.0     0.0   
SP_STATE_CODE             343644.0    25.696715    15.584441     1.0    10.0   
BENE_COUNTY_CD            343644.0   366.471174   266.006973     0.0   141.0   
BENE_HI_CVRAGE_TOT_MONS   343644.0    11.187328     2.886072     0.0    12.0   
BENE_SMI_CVRAGE_TOT_MONS  343644.0    10.840853     3.411733     0.0    12.0   
BENE_HMO_CVRAGE_TOT_MONS  343644.0     3.159421     5.189713     0.0     0.0   
PLAN_CVRG_MOS_NUM         343644.0     8.527427     5.192147     0.0     2.0   
SP_ALZHDMTA               343644.0     1.803567     0.397300     1.0     2.0   
SP_CHF                    343644.0     1.703903     0.456535     1.0     1.0   
SP_CHRNKIDN               343644.0     1

In [18]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [19]:
output_file = (
    PROCESSED_DIR /
    "beneficiary_longitudinal.csv"
)

beneficiary.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\data\processed\primary\beneficiary_longitudinal.csv


In [20]:
print("FINAL CHECK")
print("--------------------")
print("Rows:", len(beneficiary))
print("Columns:", len(beneficiary.columns))
print("Unique beneficiaries:",
      beneficiary["DESYNPUF_ID"].nunique())

print("\nRows per year:")
print(
    beneficiary["YEAR"]
    .value_counts()
    .sort_index()
)

print("\nDuplicate rows:",
      beneficiary.duplicated().sum())

print("\nMissing beneficiary IDs:",
      beneficiary["DESYNPUF_ID"].isna().sum())

FINAL CHECK
--------------------
Rows: 343644
Columns: 33
Unique beneficiaries: 116352

Rows per year:
YEAR
2008    116352
2009    114538
2010    112754
Name: count, dtype: int64

Duplicate rows: 0

Missing beneficiary IDs: 0


In [29]:
reimbursement_cols = [
    "MEDREIMB_IP",
    "BENRES_IP",
    "PPPYMT_IP",
    "MEDREIMB_OP",
    "BENRES_OP",
    "PPPYMT_OP",
    "MEDREIMB_CAR",
    "BENRES_CAR",
    "PPPYMT_CAR"
]

print(
    beneficiary[reimbursement_cols]
    .describe()
    .T
)

                 count         mean          std     min   25%    50%     75%  \
MEDREIMB_IP   343644.0  1887.001839  7145.654610 -3000.0   0.0    0.0     0.0   
BENRES_IP     343644.0   214.821391   763.951228     0.0   0.0    0.0     0.0   
PPPYMT_IP     343644.0    77.305875  1627.037876     0.0   0.0    0.0     0.0   
MEDREIMB_OP   343644.0   609.038278  1677.366102  -100.0   0.0   50.0   550.0   
BENRES_OP     343644.0   187.671253   483.310402     0.0   0.0    0.0   170.0   
PPPYMT_OP     343644.0    23.299490   355.456080     0.0   0.0    0.0     0.0   
MEDREIMB_CAR  343644.0  1117.489495  1413.605670     0.0  20.0  670.0  1630.0   
BENRES_CAR    343644.0   314.954662   392.768183     0.0   0.0  190.0   470.0   
PPPYMT_CAR    343644.0    17.506926    85.620195     0.0   0.0    0.0     0.0   

                   max  
MEDREIMB_IP   164220.0  
BENRES_IP      53096.0  
PPPYMT_IP      91000.0  
MEDREIMB_OP    50020.0  
BENRES_OP      13840.0  
PPPYMT_OP      19000.0  
MEDREIMB_CAR  

In [30]:
print(
    beneficiary[reimbursement_cols]
    .isna()
    .sum()
)

MEDREIMB_IP     0
BENRES_IP       0
PPPYMT_IP       0
MEDREIMB_OP     0
BENRES_OP       0
PPPYMT_OP       0
MEDREIMB_CAR    0
BENRES_CAR      0
PPPYMT_CAR      0
dtype: int64


In [31]:
for col in reimbursement_cols:
    print(
        col,
        "negative values =",
        (beneficiary[col] < 0).sum()
    )

MEDREIMB_IP negative values = 28
BENRES_IP negative values = 0
PPPYMT_IP negative values = 0
MEDREIMB_OP negative values = 197
BENRES_OP negative values = 0
PPPYMT_OP negative values = 0
MEDREIMB_CAR negative values = 0
BENRES_CAR negative values = 0
PPPYMT_CAR negative values = 0


Birth date after death date: 0


Records after recorded death year: 0


In [34]:
import os

print(
    "File exists:",
    os.path.exists(output_file)
)

print(
    "File size:",
    os.path.getsize(output_file) / (1024 ** 2),
    "MB"
)

File exists: True
File size: 40.28422546386719 MB
